In [12]:
import os
import h5py
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, random_split
from tqdm import tqdm

# ============================================================
# 1. Device and Hyperparameter Configuration
# ============================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

BATCH_SIZE   = 50
EPOCHS       = 15
LR           = 1e-4
WEIGHT_DECAY = 1e-5

# ============================================================
# 2. Read H5 Data and Build Dataset & DataLoader
# ============================================================
with h5py.File("atom_train_data.h5", "r") as f:
    images = f["images"][:]
    labels = f["labels"][:]

# Convert to tensor and add channel dimension: [N, 1, 7, 7]
X = torch.tensor(images, dtype=torch.float32).unsqueeze(1)
y = torch.tensor(labels, dtype=torch.long)

full_dataset = TensorDataset(X, y)

# 80% for training, 20% for testing evaluation
train_size = int(0.8 * len(full_dataset))
test_size  = len(full_dataset) - train_size
train_dataset, test_dataset = random_split(
    full_dataset, [train_size, test_size], generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Total samples: {len(full_dataset)} | Train: {train_size} | Test: {test_size}")

# ============================================================
# 3. Model Architecture Definition
# ============================================================
class SimpleAtomCNN(nn.Module):
    def __init__(self, in_channels=1, num_classes=2):
        super().__init__()
        # 7x7 -> 5x5
        self.conv1 = nn.Conv2d(in_channels, 16, kernel_size=3, padding=0)
        self.bn1   = nn.BatchNorm2d(16)
        self.relu1 = nn.ReLU(inplace=True)

        # 5x5 -> 3x3
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=0)
        self.bn2   = nn.BatchNorm2d(32)
        self.relu2 = nn.ReLU(inplace=True)

        # 3x3 -> 1x1
        self.conv3 = nn.Conv2d(32, 16, kernel_size=3, padding=0)
        self.bn3   = nn.BatchNorm2d(16)
        self.relu3 = nn.ReLU(inplace=True)

        # 16 -> 2-class classification
        self.fc = nn.Linear(16, num_classes)

    def forward(self, x):
        x = self.relu1(self.bn1(self.conv1(x)))  # [B, 16, 5, 5]
        x = self.relu2(self.bn2(self.conv2(x)))  # [B, 32, 3, 3]
        x = self.relu3(self.bn3(self.conv3(x)))  # [B, 16, 1, 1]
        x = x.squeeze(-1).squeeze(-1)             # [B, 16]
        return self.fc(x)                         # [B, 2]

# Instantiate model
model = SimpleAtomCNN().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
loss_fn   = nn.CrossEntropyLoss()

num_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {num_params:,}")

# ============================================================
# 4. Two Separate Evaluation Functions
# ============================================================
def evaluate_normal(loader):
    """
    Standard evaluation on the original (unrotated) dataset.
    """
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for x, t in loader:
            x, t = x.to(device), t.to(device)
            pred = model(x).argmax(dim=1)
            correct += (pred == t).sum().item()
            total   += t.size(0)
    return 100.0 * correct / total

def evaluate_rotated(loader):
    """
    Evaluation on rotated dataset (testing 90°, 180°, and 270° rotations to check robustness).
    """
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for x, t in loader:
            x, t = x.to(device), t.to(device)
            for rot_k in [1, 2, 3]:
                x_rot = torch.rot90(x, k=rot_k, dims=[2, 3])
                pred = model(x_rot).argmax(dim=1)
                correct += (pred == t).sum().item()
                total   += t.size(0)
    return 100.0 * correct / total

# ============================================================
# 5. Training Loop
# ============================================================
print("\n--- Training on Cold Atom Dataset ---")
for epoch in tqdm(range(EPOCHS)):
    model.train()
    for x, t in train_loader:
        x, t = x.to(device), t.to(device)
        optimizer.zero_grad()
        loss = loss_fn(model(x), t)
        loss.backward()
        optimizer.step()

    if (epoch + 1) % 5 == 0 or epoch == 0:
        normal_acc = evaluate_normal(test_loader)
        rotated_acc = evaluate_rotated(test_loader)
        print(f"Epoch {epoch+1:2d} | Normal Test Acc: {normal_acc:6.2f}% | Rotated Test Acc: {rotated_acc:6.2f}%")

# ============================================================
# 6. Final Evaluation
# ============================================================
final_normal_acc = evaluate_normal(test_loader)
final_rotated_acc = evaluate_rotated(test_loader)

print("\n" + "=" * 55)
print("Final Evaluation Summary")
print("=" * 55)
print(f"  Normal Test Acc:  {final_normal_acc:6.2f}%")
print(f"  Rotated Test Acc: {final_rotated_acc:6.2f}%")
print("=" * 55)


Using device: cpu
Total samples: 13216 | Train: 10572 | Test: 2644
Model parameters: 9,586

--- Training on Cold Atom Dataset ---


  7%|▋         | 1/15 [00:01<00:19,  1.41s/it]

Epoch  1 | Normal Test Acc:  99.81% | Rotated Test Acc:  99.79%


 33%|███▎      | 5/15 [00:05<00:12,  1.20s/it]

Epoch  5 | Normal Test Acc:  99.96% | Rotated Test Acc:  99.96%


 67%|██████▋   | 10/15 [00:12<00:06,  1.38s/it]

Epoch 10 | Normal Test Acc:  99.92% | Rotated Test Acc:  99.90%


100%|██████████| 15/15 [00:18<00:00,  1.21s/it]

Epoch 15 | Normal Test Acc: 100.00% | Rotated Test Acc: 100.00%



Final Evaluation Summary
  Normal Test Acc:  100.00%
  Rotated Test Acc: 100.00%


In [14]:
import h5py
import torch
from torch.utils.data import DataLoader, TensorDataset

# Load tricky dataset
with h5py.File("atom_tricky_data.h5", "r") as f:
    tricky_images = f["images"][:]
    tricky_labels = f["labels"][:]

X_tricky = torch.tensor(tricky_images, dtype=torch.float32).unsqueeze(1)
y_tricky = torch.tensor(tricky_labels, dtype=torch.long)

tricky_loader = DataLoader(TensorDataset(X_tricky, y_tricky), batch_size=BATCH_SIZE, shuffle=False)
print(f"Loaded {len(X_tricky)} tricky samples.")

# Evaluate using trained model
normal_acc = evaluate_normal(tricky_loader)
rotated_acc = evaluate_rotated(tricky_loader)

print(f"Normal Acc:  {normal_acc:.2f}%")
print(f"Rotated Acc: {rotated_acc:.2f}%")

Loaded 1484 tricky samples.
Normal Acc:  87.94%
Rotated Acc: 88.12%
